# Transformación ETL — Online Retail II

Flujo **Extract, Transform y Load** para las dos hojas de `02_DATOS/online_retail_II.xlsx`.

- **Extract:** verifica estructura, dimensiones, tipos y ejemplos; conserva intacto el Excel fuente y escribe un extracto raw antes de transformar.
- **Transform:** estandariza nombres/tipos, registra tratamientos de faltantes, duplicados, fechas, cancelaciones, cantidades y precios; crea variables derivadas justificadas.
- **Load:** guarda el conjunto procesado como CSV comprimido y lo vuelve a leer por bloques para verificar esquema, conteos, procedencia e importes.

No se eliminan filas ni se imputan valores. Las conversiones que no puedan hacerse sin pérdida detienen el flujo. Selecciona `.venv-1` y ejecuta **Run All**.

In [1]:
from collections import Counter
from contextlib import ExitStack
from pathlib import Path
import gzip
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

EXPECTED_SHEETS = ("Year 2009-2010", "Year 2010-2011")
SOURCE_COLUMNS = (
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country",
)
PERIOD_BY_SHEET = {
    "Year 2009-2010": "2009-2010",
    "Year 2010-2011": "2010-2011",
}
COLUMN_RENAME = {
    "Invoice": "invoice", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "invoice_date", "Price": "price",
    "Customer ID": "customer_id", "Country": "country",
    "Period": "period", "SourceSheet": "source_sheet",
}
DERIVED_COLUMNS = (
    "invoice_year", "invoice_month", "line_amount", "is_cancellation", "is_adjustment",
    "is_negative_quantity", "is_zero_price", "is_negative_price", "is_description_missing",
    "is_customer_id_missing", "is_exact_duplicate", "is_price_iqr_outlier",
    "is_quantity_iqr_outlier",
)
OUTPUT_COLUMNS = [COLUMN_RENAME[name] for name in SOURCE_COLUMNS]
OUTPUT_COLUMNS += ["period", "source_sheet", *DERIVED_COLUMNS]

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (current, *current.parents)
     if (path / "02_DATOS" / "online_retail_II.xlsx").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró 02_DATOS/online_retail_II.xlsx desde el directorio actual ni sus padres.")
SOURCE_FILE = PROJECT_ROOT / "02_DATOS" / "online_retail_II.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "03_ETL_DATA_QUALITY"
RAW_COPY_FILE = OUTPUT_DIR / "online_retail_II_extract_raw.csv.gz"
PROCESSED_FILE = OUTPUT_DIR / "online_retail_II_procesado.csv.gz"
REPORT_FILE = OUTPUT_DIR / "online_retail_II_transformacion_reporte.json"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

if not SOURCE_FILE.is_file():
    raise FileNotFoundError(f"No se encontró el archivo fuente: {SOURCE_FILE}")
source_sha256_before = sha256_file(SOURCE_FILE)
print("Fuente original:", SOURCE_FILE)
print("SHA-256:", source_sha256_before)
print("Salidas:", OUTPUT_DIR)

Fuente original: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA\02_DATOS\online_retail_II.xlsx
SHA-256: bcbe73b35f5b7babf197fb0cb983a11f5d9ff929078d4aa53d171b1f2df2e980
Salidas: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA\03_ETL_DATA_QUALITY


## 1. Extract — lectura y verificación

Las hojas se leen sin cambiar sus campos originales. Se valida el esquema esperado y se muestran dimensiones, tipos inferidos, rango de fechas y una muestra por hoja. La copia raw se escribe antes de estandarizar; el Excel fuente nunca se sobrescribe.

In [2]:
excel = pd.ExcelFile(SOURCE_FILE, engine="openpyxl")
if tuple(excel.sheet_names) != EXPECTED_SHEETS:
    raise ValueError(f"Hojas inesperadas: {tuple(excel.sheet_names)}; se esperaban {EXPECTED_SHEETS}")

raw_sheets = {}
extract_profiles = []
for sheet_name in excel.sheet_names:
    frame = pd.read_excel(excel, sheet_name=sheet_name)
    if tuple(frame.columns) != SOURCE_COLUMNS:
        raise ValueError(f"Columnas inesperadas en {sheet_name}: {tuple(frame.columns)}")
    raw_sheets[sheet_name] = frame
    extract_profiles.append({
        "Hoja": sheet_name,
        "Filas de datos": int(frame.shape[0]),
        "Columnas": int(frame.shape[1]),
        "Dimensiones Excel (incluye encabezado)": (int(frame.shape[0] + 1), int(frame.shape[1])),
        "Fecha mínima": frame["InvoiceDate"].min(),
        "Fecha máxima": frame["InvoiceDate"].max(),
    })
    print()
    print(f"Tipos originales: {sheet_name}")
    display(frame.dtypes.rename("Tipo").to_frame())
    print("Ejemplos originales:")
    display(frame.head(3))
excel.close()
display(pd.DataFrame(extract_profiles))

positive_prices = pd.concat(
    [pd.to_numeric(frame["Price"], errors="coerce") for frame in raw_sheets.values()],
    ignore_index=True,
)
positive_prices = positive_prices.loc[positive_prices > 0]
price_q1, price_q3 = positive_prices.quantile([0.25, 0.75])
price_iqr_upper = float(price_q3 + 1.5 * (price_q3 - price_q1))
absolute_quantities = pd.concat(
    [pd.to_numeric(frame["Quantity"], errors="coerce").abs() for frame in raw_sheets.values()],
    ignore_index=True,
)
quantity_q1, quantity_q3 = absolute_quantities.quantile([0.25, 0.75])
quantity_iqr_upper = float(quantity_q3 + 1.5 * (quantity_q3 - quantity_q1))
print(f"Cerca IQR de Price positivo: {price_iqr_upper:.4f}")
print(f"Cerca IQR de abs(Quantity): {quantity_iqr_upper:.4f}")


Tipos originales: Year 2009-2010


,Tipo
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
Price,float64
Customer ID,float64
Country,str


Ejemplos originales:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom



Tipos originales: Year 2010-2011


,Tipo
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
Price,float64
Customer ID,float64
Country,str


Ejemplos originales:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom


,Hoja,Filas de datos,Columnas,Dimensiones Excel (incluye encabezado),Fecha mínima,Fecha máxima
0,Year 2009-2010,525461,8,"(525462, 8)",2009-12-01 07:45:00,2010-12-09 20:01:00
1,Year 2010-2011,541910,8,"(541911, 8)",2010-12-01 08:26:00,2011-12-09 12:50:00


Cerca IQR de Price positivo: 8.5000
Cerca IQR de abs(Quantity): 23.5000


## 2. Transform — reglas y decisiones

| Regla | Tratamiento | Justificación / efecto |
|---|---|---|
| Nombres y tipos | Nombres `snake_case`; factura y código de producto como texto; cantidad y cliente como enteros anulables; fecha/hora como datetime; precio numérico. | Los identificadores no se usan como medidas; los nulos de `Customer ID` no fuerzan tipo decimal. |
| Faltantes | Mantener nulos; espacios/cadenas vacías en texto pasan a nulo. Añadir indicadores para `Description` y `Customer ID`. | No se inventan descripciones ni identidades; las filas siguen disponibles para otros análisis. |
| Duplicados | Marcar repeticiones exactas en los ocho campos fuente dentro de la misma hoja. | Conservar hasta confirmar que no sean líneas legítimas; cuantificar el posible efecto. |
| Fechas | Parsear `InvoiceDate`; un valor no parseable detiene la transformación. Derivar año y mes. | Evita conversiones silenciosas y facilita análisis temporales. |
| Facturas | Prefijo `C` se marca como cancelación; prefijo `A` como ajuste de deuda. | Son categorías observadas, no filas para descartar automáticamente. |
| Cantidades/precios | Mantener negativos y ceros; generar indicadores separados. Errores de conversión o cantidades fraccionarias detienen el flujo. | Pueden representar devoluciones, ajustes o artículos gratuitos. |
| Variables derivadas | `line_amount = quantity × price` conserva el signo; cercas IQR `Q3 + 1.5 × IQR` para precio positivo y `abs(quantity)`. | Importe firmado para rastrear el efecto aritmético; IQR solo señala extremos, no prueba error. |
| Filas | No filtrar ni deduplicar; los tratamientos son columnas de auditoría. | Diferencia de filas esperada entre entrada y salida: cero. |

In [3]:
input_rows_by_sheet = {}
sheet_metrics = {}
trimmed_counts = Counter()
parse_failures = Counter()
output_dtypes = None

with ExitStack() as stack:
    raw_output = stack.enter_context(gzip.open(RAW_COPY_FILE, "wt", encoding="utf-8-sig", newline=""))
    processed_output = stack.enter_context(gzip.open(PROCESSED_FILE, "wt", encoding="utf-8-sig", newline=""))
    for sheet_index, sheet_name in enumerate(EXPECTED_SHEETS):
        frame = raw_sheets.pop(sheet_name)
        row_count = len(frame)
        input_rows_by_sheet[sheet_name] = int(row_count)

        description_text = frame["Description"].astype("string")
        description_blank = description_text.str.strip().eq("").fillna(False)
        description_missing = frame["Description"].isna() | description_blank
        customer_missing = frame["Customer ID"].isna()
        duplicate_mask = frame.duplicated(subset=list(SOURCE_COLUMNS), keep=False)
        duplicate_excess = int(frame.duplicated(subset=list(SOURCE_COLUMNS), keep="first").sum())

        invoice_text = frame["Invoice"].astype("string").str.strip().str.upper()
        invoice_valid = invoice_text.str.fullmatch(r"(?:C|A)?\d+", na=False)
        cancellation = invoice_text.str.startswith("C", na=False)
        adjustment = invoice_text.str.startswith("A", na=False)
        stock_text = frame["StockCode"].astype("string").str.strip()
        nonstandard_stock_code = (
            stock_text.notna() & stock_text.ne("")
            & ~stock_text.str.fullmatch(r"[A-Za-z0-9]+", na=False)
        )

        quantity_numeric = pd.to_numeric(frame["Quantity"], errors="coerce")
        price_numeric = pd.to_numeric(frame["Price"], errors="coerce")
        customer_numeric = pd.to_numeric(frame["Customer ID"], errors="coerce")
        parsed_dates = pd.to_datetime(frame["InvoiceDate"], errors="coerce")
        bad_values = {
            "quantity_parse": int((frame["Quantity"].notna() & quantity_numeric.isna()).sum()),
            "quantity_fractional": int((quantity_numeric.notna() & quantity_numeric.mod(1).ne(0)).sum()),
            "price_parse": int((frame["Price"].notna() & price_numeric.isna()).sum()),
            "customer_id_parse": int((frame["Customer ID"].notna() & customer_numeric.isna()).sum()),
            "customer_id_fractional": int((customer_numeric.notna() & customer_numeric.mod(1).ne(0)).sum()),
            "date_parse": int((frame["InvoiceDate"].notna() & parsed_dates.isna()).sum()),
        }
        bad_values["price_nonfinite"] = int(
            not np.isfinite(price_numeric.dropna().to_numpy(dtype=float)).all()
        )
        parse_failures.update(bad_values)
        if any(bad_values.values()):
            raise ValueError(f"No se puede convertir sin pérdida en {sheet_name}: {bad_values}")

        trim_fields = {"Invoice": "invoice", "StockCode": "stock_code", "Description": "description", "Country": "country"}
        for source_column, target_column in trim_fields.items():
            original_text = frame[source_column].astype("string")
            trimmed_counts[target_column] += int(
                (frame[source_column].notna() & original_text.ne(original_text.str.strip()).fillna(False)).sum()
            )
        description_nontext = int(
            frame["Description"].dropna().map(lambda value: not isinstance(value, str)).sum()
        )
        negative_quantity = quantity_numeric < 0
        zero_price = price_numeric == 0
        negative_price = price_numeric < 0
        price_outlier = price_numeric > price_iqr_upper
        quantity_outlier = quantity_numeric.abs() > quantity_iqr_upper

        frame["Period"] = PERIOD_BY_SHEET[sheet_name]
        frame["SourceSheet"] = sheet_name
        frame.to_csv(raw_output, index=False, header=(sheet_index == 0), date_format="%Y-%m-%d %H:%M:%S", na_rep="")

        frame.rename(columns=COLUMN_RENAME, inplace=True)
        for column in ("invoice", "stock_code", "description", "country"):
            values = frame[column].astype("string").str.strip()
            frame[column] = values.mask(values.eq("").fillna(False), pd.NA)
        frame["quantity"] = quantity_numeric.astype("Int64")
        frame["price"] = price_numeric.astype("Float64")
        frame["customer_id"] = customer_numeric.astype("Int64")
        frame["invoice_date"] = parsed_dates
        normalized_invoice = frame["invoice"].astype("string").str.upper()
        frame["invoice_year"] = frame["invoice_date"].dt.year.astype("Int64")
        frame["invoice_month"] = frame["invoice_date"].dt.month.astype("Int64")
        frame["line_amount"] = frame["quantity"].astype("Float64") * frame["price"]
        frame["is_cancellation"] = normalized_invoice.str.startswith("C", na=False)
        frame["is_adjustment"] = normalized_invoice.str.startswith("A", na=False)
        frame["is_negative_quantity"] = negative_quantity.to_numpy()
        frame["is_zero_price"] = zero_price.to_numpy()
        frame["is_negative_price"] = negative_price.to_numpy()
        frame["is_description_missing"] = frame["description"].isna()
        frame["is_customer_id_missing"] = frame["customer_id"].isna()
        frame["is_exact_duplicate"] = duplicate_mask.to_numpy()
        frame["is_price_iqr_outlier"] = price_outlier.to_numpy()
        frame["is_quantity_iqr_outlier"] = quantity_outlier.to_numpy()
        frame = frame[OUTPUT_COLUMNS]

        sheet_metrics[sheet_name] = {
            "rows": int(row_count),
            "description_missing": int(description_missing.sum()),
            "customer_id_missing": int(customer_missing.sum()),
            "exact_duplicate_excess": duplicate_excess,
            "exact_duplicate_rows_flagged": int(duplicate_mask.sum()),
            "cancellation_rows": int(cancellation.sum()),
            "adjustment_rows": int(adjustment.sum()),
            "invalid_invoice_patterns": int((~invoice_valid).sum()),
            "nonstandard_stock_code_rows_review_only": int(nonstandard_stock_code.sum()),
            "description_nontext_values_converted_to_string": description_nontext,
            "negative_quantity_rows": int(negative_quantity.sum()),
            "zero_price_rows": int(zero_price.sum()),
            "negative_price_rows": int(negative_price.sum()),
            "price_iqr_outliers": int(price_outlier.sum()),
            "quantity_iqr_outliers": int(quantity_outlier.sum()),
        }
        if output_dtypes is None:
            output_dtypes = {column: str(dtype) for column, dtype in frame.dtypes.items()}
        frame.to_csv(processed_output, index=False, header=(sheet_index == 0), date_format="%Y-%m-%d %H:%M:%S", na_rep="")
        del frame, quantity_numeric, price_numeric, customer_numeric, parsed_dates

if raw_sheets:
    raise RuntimeError(f"Hojas no procesadas: {tuple(raw_sheets)}")
input_rows_total = sum(input_rows_by_sheet.values())
quality_counts = {
    key: sum(metrics[key] for metrics in sheet_metrics.values())
    for key in next(iter(sheet_metrics.values())) if key != "rows"
}
quality_counts["rows"] = input_rows_total
quality_counts["rows_removed"] = 0
quality_counts.update({f"{key}_failures_or_values": int(value) for key, value in parse_failures.items()})
print("Filas de entrada por hoja:")
display(pd.Series(input_rows_by_sheet, name="Filas").to_frame())
print("Conteos de tratamiento y banderas:")
display(pd.Series(quality_counts, name="Registros/casos").to_frame())
print("Valores recortados por campo:")
display(pd.Series(trimmed_counts, name="Valores con espacios externos normalizados").to_frame())
print("Archivos raw y procesado escritos sin filtrar filas:")
print(RAW_COPY_FILE)
print(PROCESSED_FILE)

Filas de entrada por hoja:


,Filas
Year 2009-2010,525461
Year 2010-2011,541910


Conteos de tratamiento y banderas:


,Registros/casos
description_missing,4382
customer_id_missing,243007
exact_duplicate_excess,12133
exact_duplicate_rows_flagged,23430
cancellation_rows,19494
adjustment_rows,6
invalid_invoice_patterns,0
nonstandard_stock_code_rows_review_only,202
description_nontext_values_converted_to_string,4
negative_quantity_rows,22950


Valores recortados por campo:


,Valores con espacios externos normalizados
invoice,0
stock_code,1
description,213035
country,0


Archivos raw y procesado escritos sin filtrar filas:
C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA\03_ETL_DATA_QUALITY\online_retail_II_extract_raw.csv.gz
C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA\03_ETL_DATA_QUALITY\online_retail_II_procesado.csv.gz


## 3. Load — relectura y verificación de integridad

Se leen ambos archivos comprimidos por bloques. Se comparan filas, columnas, distribución por hoja/periodo, faltantes y banderas con la extracción. También se comprueba que `line_amount` corresponda a `quantity × price` y que el hash del Excel original no haya cambiado.

In [4]:
expected_raw_columns = list(SOURCE_COLUMNS) + ["Period", "SourceSheet"]
raw_header = list(pd.read_csv(RAW_COPY_FILE, compression="gzip", nrows=0, encoding="utf-8-sig").columns)
if raw_header != expected_raw_columns:
    raise RuntimeError(f"Esquema de copia raw inesperado: {raw_header}")

raw_loaded_rows = 0
raw_rows_by_sheet = Counter()
raw_missing_description = 0
raw_missing_customer = 0
for chunk in pd.read_csv(
    RAW_COPY_FILE, compression="gzip", encoding="utf-8-sig", chunksize=200_000,
    dtype={"SourceSheet": "string", "Period": "string"},
):
    raw_loaded_rows += len(chunk)
    raw_rows_by_sheet.update(chunk["SourceSheet"].value_counts(dropna=False).to_dict())
    desc = chunk["Description"].astype("string")
    raw_missing_description += int((chunk["Description"].isna() | desc.str.strip().eq("").fillna(False)).sum())
    raw_missing_customer += int(chunk["Customer ID"].isna().sum())

flag_columns = [
    "is_cancellation", "is_adjustment", "is_negative_quantity", "is_zero_price",
    "is_negative_price", "is_description_missing", "is_customer_id_missing",
    "is_exact_duplicate", "is_price_iqr_outlier", "is_quantity_iqr_outlier",
]
read_dtypes = {
    "invoice": "string", "stock_code": "string", "description": "string",
    "quantity": "Int64", "price": "Float64", "customer_id": "Int64",
    "country": "string", "period": "string", "source_sheet": "string",
}
read_dtypes.update({column: "boolean" for column in flag_columns})
processed_header = list(pd.read_csv(PROCESSED_FILE, compression="gzip", nrows=0, encoding="utf-8-sig").columns)
if processed_header != OUTPUT_COLUMNS:
    raise RuntimeError(f"Esquema procesado inesperado: {processed_header}")

loaded_rows = 0
loaded_rows_by_sheet = Counter()
loaded_flags = Counter()
loaded_missing_description = 0
loaded_missing_customer = 0
period_source_mismatches = 0
line_amount_mismatches = 0
max_line_amount_error = 0.0
loaded_example = None
for chunk in pd.read_csv(
    PROCESSED_FILE, compression="gzip", encoding="utf-8-sig", chunksize=200_000,
    dtype=read_dtypes, parse_dates=["invoice_date"], float_precision="round_trip",
):
    if loaded_example is None:
        loaded_example = chunk.head(5).copy()
    loaded_rows += len(chunk)
    loaded_rows_by_sheet.update(chunk["source_sheet"].value_counts(dropna=False).to_dict())
    loaded_missing_description += int(chunk["description"].isna().sum())
    loaded_missing_customer += int(chunk["customer_id"].isna().sum())
    for column in flag_columns:
        loaded_flags[column] += int(chunk[column].fillna(False).sum())
    period_source_mismatches += int(chunk["period"].ne(chunk["source_sheet"].map(PERIOD_BY_SHEET)).sum())
    expected_amount = chunk["quantity"].astype("Float64") * chunk["price"]
    comparable = expected_amount.notna() & chunk["line_amount"].notna()
    if comparable.any():
        differences = (expected_amount[comparable] - chunk.loc[comparable, "line_amount"]).abs()
        max_line_amount_error = max(max_line_amount_error, float(differences.max()))
        line_amount_mismatches += int(differences.gt(1e-9).sum())

source_sha256_after = sha256_file(SOURCE_FILE)
checks = {
    "source_file_hash_unchanged": source_sha256_before == source_sha256_after,
    "raw_snapshot_rows_match_input": raw_loaded_rows == input_rows_total,
    "processed_rows_match_input": loaded_rows == input_rows_total,
    "raw_snapshot_source_counts_match": dict(raw_rows_by_sheet) == input_rows_by_sheet,
    "processed_source_counts_match": dict(loaded_rows_by_sheet) == input_rows_by_sheet,
    "description_missing_count_preserved": raw_missing_description == loaded_missing_description == quality_counts["description_missing"],
    "customer_id_missing_count_preserved": raw_missing_customer == loaded_missing_customer == quality_counts["customer_id_missing"],
    "period_matches_source_sheet": period_source_mismatches == 0,
    "line_amount_recomputed_exactly": line_amount_mismatches == 0,
    "no_rows_removed": quality_counts["rows_removed"] == 0 and loaded_rows == raw_loaded_rows,
}
if not all(checks.values()):
    raise RuntimeError(f"Falló la verificación de integridad: {checks}")

metric_for_flag = {
    "is_cancellation": "cancellation_rows", "is_adjustment": "adjustment_rows",
    "is_negative_quantity": "negative_quantity_rows", "is_zero_price": "zero_price_rows",
    "is_negative_price": "negative_price_rows", "is_description_missing": "description_missing",
    "is_customer_id_missing": "customer_id_missing", "is_exact_duplicate": "exact_duplicate_rows_flagged",
    "is_price_iqr_outlier": "price_iqr_outliers", "is_quantity_iqr_outlier": "quantity_iqr_outliers",
}
for column, metric in metric_for_flag.items():
    if loaded_flags[column] != quality_counts[metric]:
        raise RuntimeError(f"El conteo de {column} no coincide: {loaded_flags[column]} vs {quality_counts[metric]}")

integrity_table = pd.DataFrame(
    [(check, "OK" if passed else "FALLÓ") for check, passed in checks.items()],
    columns=["Verificación", "Resultado"],
)
display(integrity_table)
print(f"Filas entrada/raw/procesado: {input_rows_total:,} / {raw_loaded_rows:,} / {loaded_rows:,}")
print(f"Error máximo al recalcular line_amount: {max_line_amount_error:.12g}")
print("Muestra leída desde el archivo procesado:")
display(loaded_example)

,Verificación,Resultado
0,source_file_hash_unchanged,OK
1,raw_snapshot_rows_match_input,OK
2,processed_rows_match_input,OK
3,raw_snapshot_source_counts_match,OK
4,processed_source_counts_match,OK
5,description_missing_count_preserved,OK
6,customer_id_missing_count_preserved,OK
7,period_matches_source_sheet,OK
8,line_amount_recomputed_exactly,OK
9,no_rows_removed,OK


Filas entrada/raw/procesado: 1,067,371 / 1,067,371 / 1,067,371
Error máximo al recalcular line_amount: 0
Muestra leída desde el archivo procesado:


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,period,source_sheet,...,is_cancellation,is_adjustment,is_negative_quantity,is_zero_price,is_negative_price,is_description_missing,is_customer_id_missing,is_exact_duplicate,is_price_iqr_outlier,is_quantity_iqr_outlier
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009-2010,Year 2009-2010,...,False,False,False,False,False,False,False,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,Year 2009-2010,...,False,False,False,False,False,False,False,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,Year 2009-2010,...,False,False,False,False,False,False,False,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.1,13085,United Kingdom,2009-2010,Year 2009-2010,...,False,False,False,False,False,False,False,False,False,True
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-2010,Year 2009-2010,...,False,False,False,False,False,False,False,False,False,True


## 4. Diferencias, conteos y registro de reglas

Las diferencias reflejan estandarización de columnas/tipos, normalización de espacios y nuevas variables. Faltantes, duplicados, cancelaciones, devoluciones, precios no positivos y atípicos quedan marcados y cuantificados, no eliminados.

In [5]:
comparison = pd.DataFrame([
    ("Filas", input_rows_total, loaded_rows, loaded_rows - input_rows_total, "Sin eliminación ni agregación de filas."),
    ("Columnas", len(SOURCE_COLUMNS), len(OUTPUT_COLUMNS), len(OUTPUT_COLUMNS) - len(SOURCE_COLUMNS), "8 originales estandarizadas, 2 de procedencia y variables derivadas."),
    ("Description faltante", quality_counts["description_missing"], loaded_missing_description, loaded_missing_description - quality_counts["description_missing"], "Se conserva como nulo; no se imputa."),
    ("Customer ID faltante", quality_counts["customer_id_missing"], loaded_missing_customer, loaded_missing_customer - quality_counts["customer_id_missing"], "Se conserva como entero anulable; no se imputa."),
    ("Filas marcadas como duplicado exacto", quality_counts["exact_duplicate_rows_flagged"], loaded_flags["is_exact_duplicate"], loaded_flags["is_exact_duplicate"] - quality_counts["exact_duplicate_rows_flagged"], "Se marcan dentro de cada hoja y se mantienen."),
    ("Filas de cancelación C", quality_counts["cancellation_rows"], loaded_flags["is_cancellation"], loaded_flags["is_cancellation"] - quality_counts["cancellation_rows"], "Se conservan y se separan mediante bandera."),
    ("Filas de ajuste A", quality_counts["adjustment_rows"], loaded_flags["is_adjustment"], loaded_flags["is_adjustment"] - quality_counts["adjustment_rows"], "Se conservan; no se mezclan implícitamente con ventas."),
], columns=["Métrica", "Entrada", "Salida", "Diferencia", "Interpretación"])
display(comparison)
display(pd.DataFrame(
    [(field, int(count)) for field, count in trimmed_counts.items()],
    columns=["Campo", "Valores con espacios externos recortados"],
))

rules_applied = [
    "El Excel fuente no se sobrescribe; el extracto raw comprimido conserva los ocho campos originales y añade procedencia.",
    "Nombres estandarizados a snake_case; invoice y stock_code se guardan como texto.",
    "quantity y customer_id se guardan como enteros anulables; price como Float64; invoice_date como fecha/hora.",
    "Description y Customer ID faltantes se conservan; no se imputan valores.",
    "Se recortan espacios externos en texto; se cuentan cambios y cadenas vacías pasan a nulo.",
    "Duplicados exactos se marcan dentro de la hoja de origen; no se elimina ninguna fila.",
    "Prefijo C marca cancelación y A marca ajuste; se conservan ambas clases.",
    "Cantidades negativas y precios cero/negativos se conservan y se marcan por separado.",
    "Conversiones fallidas, fechas inválidas o valores fraccionarios en IDs/cantidades detienen el flujo.",
    "line_amount es quantity por price con signo; no constituye por sí solo una medida contable de ventas.",
    "Valores fuera de la cerca IQR se marcan como atípicos exploratorios; no se eliminan.",
]
summary = {
    "source_file": str(SOURCE_FILE.relative_to(PROJECT_ROOT)),
    "source_sha256_before": source_sha256_before,
    "source_sha256_after": source_sha256_after,
    "source_sheets": list(EXPECTED_SHEETS),
    "input_rows_by_sheet": input_rows_by_sheet,
    "output_rows_by_sheet": dict(loaded_rows_by_sheet),
    "rows_input": int(input_rows_total),
    "rows_output": int(loaded_rows),
    "rows_removed": 0,
    "input_columns": list(SOURCE_COLUMNS),
    "output_columns": OUTPUT_COLUMNS,
    "output_dtypes": output_dtypes,
    "quality_counts": quality_counts,
    "whitespace_trim_counts": {key: int(value) for key, value in trimmed_counts.items()},
    "parse_failures": {key: int(value) for key, value in parse_failures.items()},
    "iqr_upper_fences": {"positive_price": price_iqr_upper, "absolute_quantity": quantity_iqr_upper},
    "integrity_checks": checks,
    "rules_applied": rules_applied,
    "raw_snapshot_file": RAW_COPY_FILE.name,
    "processed_file": PROCESSED_FILE.name,
}
REPORT_FILE.write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Reporte ETL:", REPORT_FILE)
print("Extract raw:", RAW_COPY_FILE, f"({RAW_COPY_FILE.stat().st_size:,} bytes)")
print("Datos procesados:", PROCESSED_FILE, f"({PROCESSED_FILE.stat().st_size:,} bytes)")

,Métrica,Entrada,Salida,Diferencia,Interpretación
0,Filas,1067371,1067371,0,Sin eliminación ni agregación de filas.
1,Columnas,8,23,15,"8 originales estandarizadas, 2 de procedencia ..."
2,Description faltante,4382,4382,0,Se conserva como nulo; no se imputa.
3,Customer ID faltante,243007,243007,0,Se conserva como entero anulable; no se imputa.
4,Filas marcadas como duplicado exacto,23430,23430,0,Se marcan dentro de cada hoja y se mantienen.
5,Filas de cancelación C,19494,19494,0,Se conservan y se separan mediante bandera.
6,Filas de ajuste A,6,6,0,Se conservan; no se mezclan implícitamente con...


,Campo,Valores con espacios externos recortados
0,invoice,0
1,stock_code,1
2,description,213035
3,country,0


Reporte ETL: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA\03_ETL_DATA_QUALITY\online_retail_II_transformacion_reporte.json
Extract raw: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA\03_ETL_DATA_QUALITY\online_retail_II_extract_raw.csv.gz (16,125,791 bytes)
Datos procesados: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA\03_ETL_DATA_QUALITY\online_retail_II_procesado.csv.gz (20,859,278 bytes)
